# جاهز للانطلاق | Ready to begin
**SDA-DSC-211 · Tamweel Lite · دفتر 00**  
إعداد وتقديم **ميعاد المري | Meaad Al-Marri**

خصص **30–40 دقيقة** للاستعداد. ستجهّز بيئتك، تتعرف على بيانات مشروعك، وتجرب ستة أسئلة للمراجعة الذاتية. هذا النشاط **غير محسوب في الدرجة**.

**ستخرج بملفات:** `environment.json` و`data_check.json` و`runtime_checks.json` و`readiness_report.json`. هذه أدلة جاهزية؛ نتائج المشروع العلمية ستنتجها في التطبيقات اليومية.

Use a free **CPU** runtime. No payment card, GPU, API key or Drive mount is needed. Read the instructions, then choose **Runtime → Run all**. Keep a copy of the notebook and download your outputs before closing the session.

**قبل البدء:** حساب Google لفتح Colab، وحساب GitHub لحفظ عملك. افتح [دليل الاستعداد](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/READINESS_GUIDE.md).

## 1. جهّز مساحة عملك | Setup
من **Runtime → Change runtime type** اختر **CPU** (أو None إذا ظهر بهذا الاسم). شغّل الخلية التالية قبل استيراد المكتبات. سيظهر جدول الإصدارات، ثم تتحقق بقية الخلايا من البيانات والتوافق.

التنزيل مرتبط بنسخة محددة وبصمات SHA256. عند انقطاع مؤقت يعاد ثلاث مرات؛ وتُستخدم النسخة المحلية المطابقة إن كانت موجودة. إذا طلبت رسالة الخطأ إعادة التشغيل، اختر **Restart session** ثم **Run all**. لا تتجاوز فشل التحقق.

The setup installs only missing versions, limits CPU threads, and verifies downloaded files. A failed checksum stops execution.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

## 2. اعرف ما بين يديك | Data checks
التدريب يحتوي على **10,000 صف و22 خاصية**. التحدي يحتوي على **2,500 صف دون الهدف**. كل القيم اصطناعية. ثلاث خانات إضافية للتعريف والتقسيم؛ لا تدخلها في النموذج.

`default_within_90d` يعني حدث تعثر خلال 90 يومًا بعد الطلب، وليس تأخرًا بلغ 90 يومًا. اقرأ [دليل البيانات](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/data/DATA_GUIDE.md).

**الناتج المتوقع:** جدول بأحجام الملفات وعدد القيم الناقصة، ورسالة نجاح للفحوص. الفحص يتوقف إذا تغيرت الأعمدة أو البصمات أو ظهرت إجابة التحدي في ملف الخصائص.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from data_checks import load_course_data

frames, contract, data_summary = load_course_data(ROOT)
train = frames["train"]
feature_names = [item["name"] for item in contract["features"]]
target_name = contract["target"]["name"]
display(pd.DataFrame(data_summary).T.fillna("—"))
display(train[["application_id", "application_date", "income_sar", "bureau_score", target_name]].head(4))
(ROOT / "artifacts/data_check.json").write_text(json.dumps(data_summary, indent=2), encoding="utf-8")
print("Data checks passed | اجتازت البيانات الفحوص")

## 3. أربع أفكار ستستخدمها يوميًا | Four essentials

| الفكرة | معناها في مشروعك |
|---|---|
| Classification — التصنيف | تقدير الفئة 0 أو1؛ التصنيف نفسه ليس احتمالًا. |
| `predict_proba` — الاحتمال | لكل طلب احتمال بين0 و1. احتمال 0.20 ليس ضمانًا لحدوث التعثر. |
| Train / Validation / Test | تعلّم من التدريب، اختر قراراتك من التحقق، واحتفظ بالاختبار النهائي مستقلًا. |
| Confusion matrix — مصفوفة الالتباس | عدّ TP وTN وFP وFN لفهم نوع الأخطاء بعد اختيار العتبة. |

**مثال حسابي، وليس نتيجة نموذج:** إذا فاتك رصد3 حالات تعثر وأطلقت20 إنذارًا خاطئًا، فخسارة المحاكاة هي `10 × 3 + 20 = 50` وحدة. لا رسوم مالية ولا خصم50 نقطة من درجتك.

A probability is an estimate. A decision threshold converts it to a flag. In later labs you will choose the threshold using validation evidence, the simulated loss and the review capacity.

## 4. اختبر توافق الأدوات | Compatibility check
تشغّل الخلية عينة صغيرة من **300 سجل تجريبي مستقل عن بيانات مشروعك** لاختبار Logistic Regression وXGBoost وLightGBM وSHAP والمعايرة وOptuna. لا تختار نموذجًا للمشروع ولا تنتج مقارنة اليوم الأول.

**الناتج المتوقع:** ستة فحوص بحالة PASS. النجاح هنا يعني أن الأدوات تعمل معًا؛ جودة نماذجك ستحتاج إلى أدلة لاحقًا.

This is a software check, not an assessed modelling task.

In [ ]:
from readiness_checks import check_runtime

runtime_report = check_runtime(ROOT)
assert all(value == "PASS" for value in runtime_report["checks"].values())
print("Environment ready | البيئة جاهزة")

## 5. جرّب فهمك — ستة أسئلة | Six ungraded questions
اختر **A أوB أوC** لكل سؤال ثم عدّل قائمة `answers`. يمكنك تشغيل الدفتر كاملًا أولًا وترك الإجابات `None`؛ سيذكرك بإكمال المراجعة. لا تدخل هذه الإجابات في درجات الدورة.

1. خرج `predict_proba` لحالة يساوي0.20. **A:** 20 حالة مؤكدة. **B:** احتمال مقدّر20%. **C:** درجة المتدرب20.
2. أين تتعلم قيمة التعويض عن المفقود؟ **A:** كل البيانات. **B:** التحدي. **C:** جزء التدريب داخل كل طية.
3. أي عمود يحتاج انتباهًا لتسرب زمني؟ **A:** اتصالات التحصيل بعد الطلب. **B:** الدخل المعروف عند الطلب. **C:** مدة التمويل المطلوبة.
4. عميل له طلبات متعددة. **A:** افصل العملاء بين طرفي التحقق. **B:** يكفي خلط الصفوف. **C:** استخدم معرف العميل كهدف.
5. 3 حالات FN و20 حالة FP، والأوزان10 و1. الخسارة؟ **A:** 23. **B:** 50. **C:** 230.
6. أين تحفظ تقدمك قبل إغلاق Colab؟ **A:** أعتمد على الجلسة المؤقتة. **B:** لقطة شاشة فقط. **C:** نسخة Notebook والمخرجات في مستودعي أو على جهازي.

Self-check topics: probability, fold-local preprocessing, leakage, repeated customers, simulated loss, and saving your work.

In [ ]:
answers = [None, None, None, None, None, None]  # Replace None with "A", "B", or "C".
answer_key = ["B", "C", "A", "A", "B", "C"]
feedback = [
    "Probability is an estimate, not a guarantee | الاحتمال تقدير لا ضمان",
    "Fit preprocessing on training rows only | تعلّم التعويض من التدريب فقط",
    "Post-application collections are unavailable at decision time | التحصيل لاحق للقرار",
    "Separate repeated customers across validation boundaries | افصل العملاء المتكررين",
    "10 × 3 + 20 = 50 simulated units | وحدات افتراضية",
    "Save the notebook and actual outputs | احفظ الدفتر والمخرجات",
]
if len(answers) != 6 or any(a not in [None, "A", "B", "C"] for a in answers):
    raise ValueError("Use six answers: A, B, C, or None.")
answered = sum(a is not None for a in answers)
correct = sum(a == expected for a, expected in zip(answers, answer_key))
for number, (answer, expected, hint) in enumerate(zip(answers, answer_key, feedback), 1):
    status = "Not answered" if answer is None else "Correct" if answer == expected else "Review"
    print(f"{number}. {status}" + (f" — {hint}" if answer is not None else ""))
print(f"Self-check: {answered}/6 answered; {correct} correct. Not graded | غير محسوب في الدرجة")

## 6. احفظ دليل جاهزيتك | Export
ستُحفظ ملخصات البيئة والبيانات والتوافق في مجلد `artifacts`. لا تحتوي الحزمة على بيانات شخصية أو إجابات التحدي. يوجد أيضًا ملف ZIP يجمع هذه المخرجات لتسهيل تنزيلها.

من الشريط الجانبي **Files** افتح `tamweel/artifacts`، واختر **Download** للملفات أو `readiness_artifacts.zip`. افك ZIP على جهازك قبل رفع محتوياته إلى مجلد `artifacts/` في مستودعك. احفظ الدفتر من **File → Download → Download .ipynb**، ثم ارفعه إلى `notebooks/`، أو استخدم الحفظ إلى GitHub إذا كانت صلاحية حسابك جاهزة.

Keep the files as well as the notebook. A runtime can end; your saved copy is what lets you resume.

In [ ]:
import platform, zipfile

report = {"status": "READY", "dataset_version": environment["dataset_version"],
          "course_revision": COURSE_REVISION, "python": platform.python_version(),
          "elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 2),
          "data_checks": "PASS", "compatibility_checks": runtime_report["checks"],
          "self_check_answered": answered, "self_check_correct": correct,
          "self_check_is_graded": False}
if IN_COLAB:
    import resource
    report["peak_process_memory_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
(ROOT / "artifacts/readiness_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
with zipfile.ZipFile(ROOT / "artifacts/readiness_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ["environment.json", "data_check.json", "runtime_checks.json", "readiness_report.json"]:
        bundle.write(ROOT / "artifacts" / filename, filename)
print("READY — احفظ ملفات artifacts ونسخة الدفتر")
print(f"Elapsed: {report['elapsed_seconds']} seconds | CPU | {len(train):,} training rows")

## قبل أن تغلق الدفتر | Before you leave
- [ ] رأيت رسالة `Environment ready` دون خطأ.
- [ ] فهمت الفرق بين الخصائص والهدف ومعرّفات التقسيم.
- [ ] راجعت الأسئلة الستة؛ نتيجتها لا تدخل في الدرجة.
- [ ] حفظت الدفتر والمخرجات خارج الجلسة المؤقتة.

**تأمل قصير:** لماذا لا يكفي أن يكون ترتيب التواريخ صحيحًا إذا لم يكتمل رصد نتيجة طلب التدريب؟ دوّن إجابتك بكلماتك.

**للتوسع:** أعد تشغيل الدفتر بعد Restart session. يجب أن يجتاز البيانات نفسها دون اعتماد على متغيرات من تشغيل سابق.

**إذا تعثرت:** اقرأ رسالة الخطأ أولًا. فشل التنزيل: أعد خلية الإعداد. اختلاف إصدار مستورد: أعد الجلسة وشغّل الإعداد أولًا. اختلاف بصمة: استرجع الملف الأصلي دون تعديل. حدود الخدمة: احفظ عملك وأعد لاحقًا؛ لا تشترِ موارد.

**خطوتك التالية:** احتفظ بهذا الدفتر مرجعًا، ثم افتح تطبيق اليوم الأول عند إتاحته في [بوابة الدورة](https://almiyead-rgb.github.io/advanced-machine-learning-methods-sda-dsc-211/).